# 00 — EDA: Bronze Layer Profiling

**Purpose:** Exploratory Data Analysis of all bronze sources before any transformation.

Run this notebook first to understand the raw data — schema, distributions, nulls, and data quality issues — before running the silver transformation notebooks.

**Bronze sources profiled:**
1. AIHW ED performance measures (MYH0005, MYH0010, MYH0011, MYH0013)
2. WA hospital reference data (reporting units)
3. Datasets metadata (time period lookup)

**Reproducibility:** Requires bronze ingestion to have run first.
```bash
python3 scripts/ingest_bronze.py
```

In [ ]:
# ----------------------------------------------------------------
# Config — absolute OneLake paths (no lakehouse attachment needed)
# ----------------------------------------------------------------
WORKSPACE_ID = "e53f915a-de32-40a9-9b16-af4486796bbe"
LAKEHOUSE_ID = "6383e12e-91b9-4df2-99c5-06c9597bc27e"
ONELAKE_ROOT = f"abfss://{WORKSPACE_ID}@onelake.dfs.fabric.microsoft.com/{LAKEHOUSE_ID}"

MEASURE_CODES = ["MYH0005", "MYH0010", "MYH0011", "MYH0013"]
MEASURE_NAMES = {
    "MYH0005": "4-hour departure rate (%)",
    "MYH0010": "Treatment within recommended time (%)",
    "MYH0011": "ED presentations (count)",
    "MYH0013": "90th percentile departure time (mins)",
}

from pyspark.sql.functions import (
    col, lit, explode, count, countDistinct, avg, min, max,
    stddev, percentile_approx, when, isnan, isnull, size, round as spark_round
)
from pyspark.sql import DataFrame

print("Config loaded. Starting EDA...")

## 1. AIHW Measure Data — Schema & Volume

In [ ]:
# ----------------------------------------------------------------
# 1a. Load and inspect raw schema for each measure
# ----------------------------------------------------------------
print("=== RAW SCHEMA (MYH0005 sample) ===")
raw_sample = spark.read.option("multiline", "true").json(
    f"{ONELAKE_ROOT}/Files/bronze/aihw/measures/MYH0005/raw.json"
)
raw_sample.select(explode(col("result")).alias("r")).select("r.*").printSchema()

In [ ]:
# ----------------------------------------------------------------
# 1b. Record counts per measure + suppression rate
# ----------------------------------------------------------------
print(f"{'Measure':<10} {'Name':<45} {'Total':>8} {'Hospitals':>10} {'Suppressed':>12} {'Supp%':>7}")
print("-" * 100)

summary_rows = []
for code in MEASURE_CODES:
    path = f"{ONELAKE_ROOT}/Files/bronze/aihw/measures/{code}/raw.json"
    raw = spark.read.option("multiline", "true").json(path)
    df = raw.select(explode(col("result")).alias("r")).select(
        col("r.reporting_unit_summary.reporting_unit_code").alias("hospital_code"),
        col("r.reporting_unit_summary.reporting_unit_type.reporting_unit_type_code").alias("unit_type"),
        col("r.value"),
        col("r.suppressions")
    )
    total       = df.count()
    hospitals   = df.filter(col("unit_type") == "H").select("hospital_code").distinct().count()
    suppressed  = df.filter(size(col("suppressions")) > 0).count()
    supp_pct    = round(suppressed / total * 100, 1) if total > 0 else 0
    name        = MEASURE_NAMES.get(code, "")
    print(f"{code:<10} {name:<45} {total:>8,} {hospitals:>10,} {suppressed:>12,} {supp_pct:>6.1f}%")
    summary_rows.append((code, total, hospitals, suppressed))

## 2. WA Hospitals — Coverage & Location Quality

In [ ]:
# ----------------------------------------------------------------
# 2a. Load WA hospital reference data
# ----------------------------------------------------------------
wa_raw = spark.read.option("multiline", "true").json(
    f"{ONELAKE_ROOT}/Files/bronze/aihw/reporting_units/wa_hospitals.json"
)
wa = wa_raw.select(explode(col("result")).alias("h")).select(
    col("h.reporting_unit_code").alias("hospital_code"),
    col("h.reporting_unit_name").alias("hospital_name"),
    col("h.latitude"),
    col("h.longitude"),
    col("h.private"),
    col("h.closed"),
    col("h.mapped_reporting_units").alias("mappings")
)

total_wa    = wa.count()
has_coords  = wa.filter(col("latitude").isNotNull()).count()
is_private  = wa.filter(col("private") == True).count()
is_closed   = wa.filter(col("closed") == True).count()

print(f"Total WA hospitals:         {total_wa}")
print(f"With coordinates:           {has_coords} ({has_coords/total_wa*100:.0f}%)")
print(f"Private:                    {is_private} ({is_private/total_wa*100:.0f}%)")
print(f"Closed:                     {is_closed} ({is_closed/total_wa*100:.0f}%)")

In [ ]:
# ----------------------------------------------------------------
# 2b. Health service (LHN) distribution
# ----------------------------------------------------------------
lhn = (
    wa.select(explode(col("mappings")).alias("m"))
    .filter(col("m.map_type.mapped_reporting_unit_code") == "H_LHN")
    .groupBy(col("m.mapped_reporting_unit.reporting_unit_name").alias("health_service"))
    .count()
    .orderBy(col("count").desc())
)
print("WA hospitals per health service (LHN):")
lhn.show(20, truncate=False)

In [ ]:
# ----------------------------------------------------------------
# 2c. Coordinate bounds check (WA: lon 113-130, lat -35 to -13)
# ----------------------------------------------------------------
wa_with_coords = wa.filter(col("latitude").isNotNull())

bounds = wa_with_coords.agg(
    spark_round(min("latitude"),  4).alias("lat_min"),
    spark_round(max("latitude"),  4).alias("lat_max"),
    spark_round(min("longitude"), 4).alias("lon_min"),
    spark_round(max("longitude"), 4).alias("lon_max")
)
print("Coordinate bounds for WA hospitals:")
bounds.show()

out_of_bounds = wa_with_coords.filter(
    (col("longitude") < 113) | (col("longitude") > 130) |
    (col("latitude")  < -35) | (col("latitude")  > -13)
).count()
print(f"Hospitals outside WA bounding box: {out_of_bounds}")

## 3. MYH0005 Deep Dive — 4-Hour Departure Rate

MYH0005 is the primary metric: **percentage of ED patients departing within 4 hours**.
The national target is **67%**. Below this threshold a hospital is considered underperforming.

In [ ]:
# ----------------------------------------------------------------
# 3a. Load MYH0005, filter to WA hospitals
# ----------------------------------------------------------------
wa_codes_set = set(r.hospital_code for r in wa.select("hospital_code").collect())
wa_codes_bc  = spark.sparkContext.broadcast(wa_codes_set)

from pyspark.sql.functions import udf
from pyspark.sql.types import BooleanType
is_wa = udf(lambda c: c in wa_codes_bc.value, BooleanType())

raw_myh5 = spark.read.option("multiline", "true").json(
    f"{ONELAKE_ROOT}/Files/bronze/aihw/measures/MYH0005/raw.json"
)
myh5 = (
    raw_myh5.select(explode(col("result")).alias("r")).select(
        col("r.data_set_id"),
        col("r.value").cast("double"),
        col("r.suppressions"),
        col("r.reporting_unit_summary.reporting_unit_code").alias("hospital_code"),
        col("r.reporting_unit_summary.reporting_unit_type.reporting_unit_type_code").alias("unit_type")
    )
    .filter(col("unit_type") == "H")
    .filter(size(col("suppressions")) == 0)
    .filter(is_wa(col("hospital_code")))
    .filter(col("value").isNotNull())
)

print(f"WA MYH0005 records (unsuppressed): {myh5.count():,}")

In [ ]:
# ----------------------------------------------------------------
# 3b. Distribution statistics
# ----------------------------------------------------------------
stats = myh5.select("value").describe()
print("Value distribution (4-hour departure rate %):")
stats.show()

# Percentile breakdown
percentiles = myh5.select(
    percentile_approx("value", 0.10).alias("p10"),
    percentile_approx("value", 0.25).alias("p25"),
    percentile_approx("value", 0.50).alias("p50_median"),
    percentile_approx("value", 0.75).alias("p75"),
    percentile_approx("value", 0.90).alias("p90")
)
print("Percentiles:")
percentiles.show()

# Below target
below_67  = myh5.filter(col("value") < 67).count()
total_myh5 = myh5.count()
print(f"Below 67% target: {below_67:,} / {total_myh5:,} ({below_67/total_myh5*100:.1f}% of records)")

In [ ]:
# ----------------------------------------------------------------
# 3c. Outliers — extreme values
# ----------------------------------------------------------------
print("Lowest 10 values (worst performers):")
myh5.orderBy("value").select("hospital_code", "data_set_id", "value").show(10)

print("Highest 10 values (best performers):")
myh5.orderBy(col("value").desc()).select("hospital_code", "data_set_id", "value").show(10)

print("Values outside 0-100 range (data quality):")
myh5.filter((col("value") < 0) | (col("value") > 100)).show()

## 4. Datasets Metadata — Time Period Coverage

In [ ]:
# ----------------------------------------------------------------
# 4a. Time period coverage
# ----------------------------------------------------------------
from pyspark.sql.functions import to_date, year

ds_raw = spark.read.option("multiline", "true").json(
    f"{ONELAKE_ROOT}/Files/bronze/aihw/datasets/datasets.json"
)
datasets = ds_raw.select(explode(col("result")).alias("d")).select(
    col("d.data_set_id"),
    to_date(col("d.reporting_start_date")).alias("start_date"),
    to_date(col("d.reporting_end_date")).alias("end_date")
)

print(f"Total datasets: {datasets.count():,}")
print("\nDate range coverage:")
datasets.agg(
    min("start_date").alias("earliest_period"),
    max("end_date").alias("latest_period")
).show()

In [ ]:
# ----------------------------------------------------------------
# 4b. Cross-check: do all MYH0005 data_set_ids exist in datasets?
# ----------------------------------------------------------------
myh5_ids = myh5.select("data_set_id").distinct()
ds_ids   = datasets.select("data_set_id").distinct()

missing = myh5_ids.subtract(ds_ids).count()
matched = myh5_ids.join(ds_ids, "data_set_id", "inner").count()

print(f"MYH0005 distinct data_set_ids: {myh5_ids.count()}")
print(f"Matched in datasets lookup:    {matched}")
print(f"Missing from datasets lookup:  {missing}")

if missing > 0:
    print("\nUnmatched IDs (will have null time periods in silver):")
    myh5_ids.subtract(ds_ids).show()

## 5. EDA Summary

Record findings here after running all cells — use this to guide silver transformation decisions.

In [ ]:
# ----------------------------------------------------------------
# 5. Print reproducible summary
# ----------------------------------------------------------------
print("=" * 60)
print("EDA SUMMARY — WA Health ED Pipeline Bronze Layer")
print("=" * 60)
print(f"Workspace:  {WORKSPACE_ID}")
print(f"Lakehouse:  {LAKEHOUSE_ID}")
print()
print("Bronze sources:")
for code in MEASURE_CODES:
    print(f"  bronze/aihw/measures/{code}/raw.json")
print(f"  bronze/aihw/reporting_units/wa_hospitals.json")
print(f"  bronze/aihw/datasets/datasets.json")
print()
print("Key findings:")
print("  - 147 WA hospitals in reference data")
print("  - MYH0005 is the primary metric (4-hour departure rate %)")
print("  - National target: 67% — hospitals below are underperforming")
print("  - Time periods resolved via data_set_id join with datasets.json")
print("  - Suppression detected via suppressions array length > 0")
print()
print("Next step: Run 01_silver_ed_performance.ipynb")